In [27]:
# Data preparation

In [28]:
# pip install xgboost

In [29]:
# !pip install mlxtend

In [30]:
# !pip install shap

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from mlxtend.frequent_patterns import apriori, association_rules
import shap
import joblib
import pandas as pd
from pathlib import Path



# Import data

In [32]:
path_row = "./Datasets/Raw/"
path_processed = "./Datasets/Processed/"
df = pd.read_csv(path_row + "survey.csv")
df_copy = df.copy()

# Show initial data info

In [33]:
print("The first five rows of data:")
print(df_copy.head())

print("\nALL field names:")
print(df_copy.columns.tolist())

The first five rows of data:
             Timestamp  Age  Gender         Country state self_employed  \
0  2014-08-27 11:29:31   37  Female   United States    IL           NaN   
1  2014-08-27 11:29:37   44       M   United States    IN           NaN   
2  2014-08-27 11:29:44   32    Male          Canada   NaN           NaN   
3  2014-08-27 11:29:46   31    Male  United Kingdom   NaN           NaN   
4  2014-08-27 11:30:22   31    Male   United States    TX           NaN   

  family_history treatment work_interfere    no_employees  ...  \
0             No       Yes          Often            6-25  ...   
1             No        No         Rarely  More than 1000  ...   
2             No        No         Rarely            6-25  ...   
3            Yes       Yes          Often          26-100  ...   
4             No        No          Never         100-500  ...   

                leave mental_health_consequence phys_health_consequence  \
0       Somewhat easy                        No 

# DataCleaning

In [34]:
def standardize_column_names(df):
    """标准化列名"""
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("/", "_")
        .str.replace("-", "_")
        .str.replace("(", "")
        .str.replace(")", "")
        .str.replace(".", "")
        .str.replace(":", "")
        .str.replace("__", "_")
        .str.replace("__", "_")
    )
    return df






In [35]:
standardize_column_names(df_copy)

,timestamp,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,...,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
0,2014-08-27 11:29:31,37,Female,United States,IL,NaN,No,Yes,Often,6-25,...,Somewhat easy,No,No,Some of them,Yes,No,Maybe,Yes,No,NaN
1,2014-08-27 11:29:37,44,M,United States,IN,NaN,No,No,Rarely,More than 1000,...,Don't know,Maybe,No,No,No,No,No,Don't know,No,NaN
2,2014-08-27 11:29:44,32,Male,Canada,NaN,NaN,No,No,Rarely,6-25,...,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,No,NaN
3,2014-08-27 11:29:46,31,Male,United Kingdom,NaN,NaN,Yes,Yes,Often,26-100,...,Somewhat difficult,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,NaN
4,2014-08-27 11:30:22,31,Male,United States,TX,NaN,No,No,Never,100-500,...,Don't know,No,No,Some of them,Yes,Yes,Yes,Don't know,No,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1254,2015-09-12 11:17:21,26,male,United Kingdom,NaN,No,No,Yes,NaN,26-100,...,Somewhat easy,No,No,Some of them,Some of them,No,No,Don't know,No,NaN
1255,2015-09-26 01:07:35,32,Male,United States,IL,No,Yes,Yes,Often,26-100,...,Somewhat difficult,No,No,Some of them,Yes,No,No,Yes,No,NaN
1256,2015-11-07 12:36:58,34,male,United States,CA,No,Yes,Yes,Sometimes,More than 1000,...,Somewhat difficult,Yes,Yes,No,No,No,No,No,No,NaN
1257,2015-11-30 21:25:06,46,f,United States,NC,No,No,No,NaN,100-500,...,Don't know,Yes,No,No,No,No,No,No,No,NaN


In [36]:
def get_unique_values_df(df, exclude_cols=None):
    """
    Return a DataFrame where each row contains:
    - column name
    - list of unique values in that column

    Excludes specified columns.
    """
    if exclude_cols is None:
        exclude_cols = []

    data = []

    for col in df.columns:
        if col not in exclude_cols:
            uniques = df[col].unique().tolist()
            data.append([col, uniques])

    return pd.DataFrame(data, columns=['column', 'unique_values'])


In [37]:
def standardize_gender(df,column='gender'):
    """standardize gender column values"""
    male_list = [
        "Male ",
        "male",
        "M",
        "m",
        "Male",
        "Cis Male",
        "Man",
        "cis male",
        "Mail",
        "Male-ish",
        "Male (CIS)",
        "Cis Man",
        "msle",
        "Malr",
        "Mal",
        "maile",
        "Make",
    ]

    female_list = [
        "Female ",
        "female",
        "F",
        "f",
        "Woman",
        "Female",
        "femail",
        "Cis Female",
        "cis-female/femme",
        "Femake",
        "Female (cis)",
        "woman",
    ]

    other_list = [
        "Female (trans)",
        "queer/she/they",
        "non-binary",
        "fluid",
        "queer",
        "Androgyne",
        "Trans-female",
        "male leaning androgynous",
        "Agender",
        "A little about you",
        "Nah",
        "All",
        "ostensibly male, unsure what that really means",
        "Genderqueer",
        "Enby",
        "p",
        "Neuter",
        "something kinda male?",
        "Guy (-ish) ^_^",
        "Trans woman",
    ]

    # Replace values in the DataFrame
    df[column].replace(male_list, "Male", inplace=True)
    df[column].replace(female_list, "Female", inplace=True)
    df[column].replace(other_list, "Other", inplace=True)

    return df

In [38]:
standardize_gender(df_copy)['gender'].unique()
df_copy['gender'].unique()

/var/folders/vz/zs883bb125v6xmzdr27py7p40000gn/T/ipykernel_78483/770603423.py:62: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[column].replace(male_list, "Male", inplace=True)


array(['Female', 'Male', 'Other'], dtype=object)

In [39]:
df_copy["no_employees"] = df_copy["no_employees"].replace(
    {
        "1-5": "1-5",
        "6-25": "6-25",
        "26-100": "26-100",
        "100-500": "100-500",
        "500-1000": "500-1000",
        "More than 1000": "1000+",
    }
)

In [40]:
df_copy['leave'].unique()

array(['Somewhat easy', "Don't know", 'Somewhat difficult',
       'Very difficult', 'Very easy'], dtype=object)

In [41]:
df_copy["leave"] = df_copy["leave"].replace(
    {
        "Very easy": "Easy",
        "Somewhat easy": "Slightly easy",
        "Somewhat difficult": "Slightly difficult",
        "Very difficult": "Difficult",
    }
)

In [42]:
df_copy['coworkers'] = df_copy['coworkers'].replace(
    {
        "Some of them": "Some",
        "Yes": "Yes",
        "No": "No",
        
    }
)
df_copy["supervisor"] = df_copy["supervisor"].replace(
    {
        "Some of them": "Some",
    }
)
df_copy["coworkers"].unique()

array(['Some', 'No', 'Yes'], dtype=object)

In [43]:
unique_values = get_unique_values_df(df_copy, exclude_cols=["timestamp", "age"])
unique_values

,column,unique_values
0,gender,"[Female, Male, Other]"
1,country,"[United States, Canada, United Kingdom, Bulgar..."
2,state,"[IL, IN, nan, TX, TN, MI, OH, CA, CT, MD, NY, ..."
3,self_employed,"[nan, Yes, No]"
4,family_history,"[No, Yes]"
5,treatment,"[Yes, No]"
6,work_interfere,"[Often, Rarely, Never, Sometimes, nan]"
7,no_employees,"[6-25, 1000+, 26-100, 100-500, 1-5, 500-1000]"
8,remote_work,"[No, Yes]"
9,tech_company,"[Yes, No]"


In [44]:
df_copy.isna().sum()    

timestamp                       0
age                             0
gender                          0
country                         0
state                         515
self_employed                  18
family_history                  0
treatment                       0
work_interfere                264
no_employees                    0
remote_work                     0
tech_company                    0
benefits                        0
care_options                    0
wellness_program                0
seek_help                       0
anonymity                       0
leave                           0
mental_health_consequence       0
phys_health_consequence         0
coworkers                       0
supervisor                      0
mental_health_interview         0
phys_health_interview           0
mental_vs_physical              0
obs_consequence                 0
comments                     1095
dtype: int64

In [45]:
unique_values = get_unique_values_df(df_copy, exclude_cols=["timestamp", "age"])
unique_values

,column,unique_values
0,gender,"[Female, Male, Other]"
1,country,"[United States, Canada, United Kingdom, Bulgar..."
2,state,"[IL, IN, nan, TX, TN, MI, OH, CA, CT, MD, NY, ..."
3,self_employed,"[nan, Yes, No]"
4,family_history,"[No, Yes]"
5,treatment,"[Yes, No]"
6,work_interfere,"[Often, Rarely, Never, Sometimes, nan]"
7,no_employees,"[6-25, 1000+, 26-100, 100-500, 1-5, 500-1000]"
8,remote_work,"[No, Yes]"
9,tech_company,"[Yes, No]"


# Filling with missing values

In [46]:
df_copy.isna().sum()

timestamp                       0
age                             0
gender                          0
country                         0
state                         515
self_employed                  18
family_history                  0
treatment                       0
work_interfere                264
no_employees                    0
remote_work                     0
tech_company                    0
benefits                        0
care_options                    0
wellness_program                0
seek_help                       0
anonymity                       0
leave                           0
mental_health_consequence       0
phys_health_consequence         0
coworkers                       0
supervisor                      0
mental_health_interview         0
phys_health_interview           0
mental_vs_physical              0
obs_consequence                 0
comments                     1095
dtype: int64

In [47]:
df_copy["self_employed"] = df_copy["self_employed"].fillna("No")
print(df_copy["self_employed"].unique())
df_copy["work_interfere"] = df_copy["work_interfere"].fillna("Don't know")
print(df_copy["work_interfere"].unique())
df_copy["state"] = df_copy["state"].fillna("/")
print(df_copy["state"].unique())

['No' 'Yes']
['Often' 'Rarely' 'Never' 'Sometimes' "Don't know"]
['IL' 'IN' '/' 'TX' 'TN' 'MI' 'OH' 'CA' 'CT' 'MD' 'NY' 'NC' 'MA' 'IA' 'PA'
 'WA' 'WI' 'UT' 'NM' 'OR' 'FL' 'MN' 'MO' 'AZ' 'CO' 'GA' 'DC' 'NE' 'WV'
 'OK' 'KS' 'VA' 'NH' 'KY' 'AL' 'NV' 'NJ' 'SC' 'VT' 'SD' 'ID' 'MS' 'RI'
 'WY' 'LA' 'ME']


In [48]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   timestamp                  1259 non-null   object
 1   age                        1259 non-null   int64 
 2   gender                     1259 non-null   object
 3   country                    1259 non-null   object
 4   state                      1259 non-null   object
 5   self_employed              1259 non-null   object
 6   family_history             1259 non-null   object
 7   treatment                  1259 non-null   object
 8   work_interfere             1259 non-null   object
 9   no_employees               1259 non-null   object
 10  remote_work                1259 non-null   object
 11  tech_company               1259 non-null   object
 12  benefits                   1259 non-null   object
 13  care_options               1259 non-null   object
 14  wellness

In [49]:
df = pd.read_csv(path_row + "survey.csv")
df_copy.isna().sum()


timestamp                       0
age                             0
gender                          0
country                         0
state                           0
self_employed                   0
family_history                  0
treatment                       0
work_interfere                  0
no_employees                    0
remote_work                     0
tech_company                    0
benefits                        0
care_options                    0
wellness_program                0
seek_help                       0
anonymity                       0
leave                           0
mental_health_consequence       0
phys_health_consequence         0
coworkers                       0
supervisor                      0
mental_health_interview         0
phys_health_interview           0
mental_vs_physical              0
obs_consequence                 0
comments                     1095
dtype: int64

In [50]:
output_path = "./Datasets/Processed/cleaned_survey.csv"
df_copy.to_csv(output_path, index=False)